<a href="https://colab.research.google.com/github/pxtroniwnl/barcelona-de-indias-time-serie/blob/main/precipitacioonIDEAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Precipitación — estaciones IDEAM cercanas al ROI de la laguna

**Fuente:** CSV del IDEAM, departamento de Bolívar, sensor de precipitación (`0240`, unidad `mm`).

**Objetivo:** identificar las estaciones pluviométricas más cercanas al área de estudio
(laguna peri-urbana en el norte de Cartagena), reconstruir su serie de precipitación a paso
horario / diario / mensual con control de cobertura, y dejarla lista para cruzarla con la serie
satelital de cobertura de *Eichhornia crassipes*.

## Estructura

| Sección | Contenido |
|---|---|
| 1 | Configuración, constantes y autenticación GEE |
| 2 | Carga del CSV crudo (`df_raw`, inmutable) |
| 3 | Limpieza y control de calidad |
| 4 | Catálogo de estaciones |
| 5 | Geometría: ROI y cajas anidadas |
| 6 | Clasificación espacial y distancias |
| 7 | Mapa |
| 8 | Selección de estaciones y serie temporal |
| 9 | ¿Acumulado por intervalo o acumulado corrido? |
| 10 | Paso de muestreo real y huecos del registro |
| 11 | Descriptores sobre la serie nativa |
| 12 | Visualización |
| 13 | Precipitación antecedente (enlace con la serie satelital) |
| 14 | Exportación |
| 15 | Anexo opcional — agregación temporal |
| 16 | Limitaciones |


## Principio de organización

Cada celda es **idempotente**: puede reejecutarse sin corromper el estado. El crudo (`df_raw`)
nunca se sobrescribe; toda transformación usa `.copy()` y devuelve un objeto nuevo. Esto permite
`Runtime > Run all` sin sorpresas.


## 1. Configuración y autenticación


In [ ]:
# Instalación (solo en Colab, la primera vez)
# !pip install -q pandas geemap earthengine-api

import csv
import io
import math
from pathlib import Path

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- rutas
# Ajusta el nombre al archivo real. Si no existe, la celda de carga busca por patrón en /content.
RUTA_CSV = Path("/content/Precipitación_20260818_SOLO_BOLIVAR.csv")
PATRON_BUSQUEDA = "*recipitacion*.csv"       # respaldo si la ruta anterior no existe
DIR_SALIDA = Path("/content")                # dónde se escriben los CSV derivados

# ---------------------------------------------------------------- parseo
FORMATO_FECHA = "%Y %b %d %I:%M:%S %p"  # ej. "2019 Aug 04 01:40:00 AM"

# Columnas que deben conservarse como texto (los ceros a la izquierda son parte del ID)
DTYPES_TEXTO = {"CodigoEstacion": "string", "CodigoSensor": "string"}

COLS_TEXTO = [
    "CodigoEstacion", "CodigoSensor", "NombreEstacion", "Departamento",
    "Municipio", "ZonaHidrografica", "DescripcionSensor", "UnidadMedida",
]

# ---------------------------------------------------------------- control de calidad
# Rango físicamente plausible para la lámina acumulada EN UN INTERVALO DE MUESTREO (no diaria).
# El límite inferior es 0.0: en precipitación el cero es un dato válido, no un faltante.
# El superior atrapa centinelas (-9999) y desbordes del pluviómetro de balancín.
# Elección empírica, NO un estándar del IDEAM: revisar la distribución real antes de fijarlo.
RANGO_PP_VALIDO = (0.0, 100.0)

# Los valores por encima de este umbral se conservan pero se marcan para inspección manual.
UMBRAL_PP_SOSPECHOSO = 25.0   # mm en un solo intervalo de muestreo

# Umbral para considerar que un INTERVALO registró lluvia (resolución típica del balancín)
MM_INTERVALO_HUMEDO = 0.1

# ---------------------------------------------------------------- huecos
# Un salto entre registros consecutivos mayor que FACTOR_HUECO x el paso de muestreo se
# considera interrupción del registro, no un intervalo normal.
FACTOR_HUECO = 3

# Paso de muestreo por defecto si no puede estimarse (estación con un solo registro en el año)
PASO_FALLBACK_S = 600.0   # 10 min, el más común en la red automática

# ---------------------------------------------------------------- geometría
# Vértices del ROI (laguna). Longitud primero, luego latitud.
ROI_COORDS = [
    [-75.476052, 10.517524],
    [-75.476117, 10.518747],
    [-75.473158, 10.519223],
    [-75.470516, 10.525108],
    [-75.469572, 10.524876],
    [-75.471686, 10.518916],
    [-75.468394, 10.517219],
    [-75.468952, 10.516459],
]

# Factores de escalado de las cajas anidadas.
# NOTA: el factor multiplica el LADO, no el área. Una caja 2x tiene 4x el área.
FACTORES = [1, 2, 3, 4]
COLORES_CAJA = {1: "FF0000", 2: "FF8800", 3: "FFFF00", 4: "00FF00"}

# Número de estaciones a seleccionar para el análisis
N_ESTACIONES = 3

# Tolerancia para considerar dos registros el mismo sitio físico (grados ≈ 110 m)
TOL_SITIO = 3  # decimales de redondeo

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# ---------------------------------------------------------------- Earth Engine
PROYECTO_GEE = "proyecto1-504013"

ee.Authenticate(auth_mode="notebook")
ee.Initialize(project=PROYECTO_GEE)
print("GEE listo:", ee.String("ok").getInfo())


## 2. Carga del CSV crudo

Los volcados del IDEAM circulan en dos formatos: CSV estándar y CSV con una capa extra de comillas
(cada línea completa envuelta como un único campo, con las comillas internas duplicadas). El cargador
**detecta cuál es** leyendo la primera fila, en vez de asumirlo: si el desescapado se aplica a un
archivo que no lo necesita, pandas recibe una sola columna y todo lo que sigue falla.

`df_raw` **no se modifica nunca**. Para volver a un estado limpio basta reejecutar esta celda.


In [ ]:
def resolver_ruta(ruta: Path, patron: str = PATRON_BUSQUEDA) -> Path:
    """Devuelve la ruta si existe; si no, busca por patrón en su directorio."""
    if ruta.exists():
        return ruta
    candidatos = sorted(ruta.parent.glob(patron))
    if not candidatos:
        raise FileNotFoundError(
            f"No existe {ruta} ni ningún archivo que case con {patron} en {ruta.parent}"
        )
    print(f"[carga] {ruta.name} no existe; se usa {candidatos[0].name}")
    return candidatos[0]


def cargar_crudo(ruta: Path) -> pd.DataFrame:
    """Carga el CSV del IDEAM, desescapando el doble entrecomillado solo si está presente."""
    with open(ruta, encoding="utf-8") as f:
        texto = f.read()

    primera = next(csv.reader(io.StringIO(texto)))
    if len(primera) == 1 and "," in primera[0]:
        # Cada fila es un único campo que a su vez contiene un CSV: hay que desenvolverlo.
        lineas = [fila[0] for fila in csv.reader(io.StringIO(texto)) if fila]
        texto = "\n".join(lineas)
        print("[carga] doble entrecomillado detectado → desescapado")
    else:
        print("[carga] CSV estándar")

    return pd.read_csv(io.StringIO(texto), dtype=DTYPES_TEXTO)


RUTA_CSV = resolver_ruta(RUTA_CSV)
df_raw = cargar_crudo(RUTA_CSV)
print(f"{len(df_raw):,} filas × {df_raw.shape[1]} columnas")
df_raw.head(3)


In [ ]:
# Diagnóstico del crudo: valores categóricos, unidades y distribución del valor observado
for col in ["UnidadMedida", "DescripcionSensor", "CodigoSensor", "Departamento"]:
    vals = sorted(df_raw[col].dropna().unique().tolist())
    print(f"{col:20s} ({len(vals):>3}): {vals[:6]}{' …' if len(vals) > 6 else ''}")

unidades = set(df_raw["UnidadMedida"].dropna().unique())
if unidades != {"mm"}:
    print(f"\n⚠  Unidades mezcladas: {unidades}. Homogeneizar antes de sumar.")

print()
_v = pd.to_numeric(df_raw["ValorObservado"], errors="coerce")
print("Distribución de ValorObservado (crudo)")
print(_v.describe(percentiles=[0.5, 0.9, 0.99, 0.999, 0.9999]).round(3))
print(f"\nceros exactos : {(_v == 0).sum():,}  ({(_v == 0).mean() * 100:.1f} %)")
print(f"negativos     : {(_v < 0).sum():,}")
print(f"> {RANGO_PP_VALIDO[1]:g} mm   : {(_v > RANGO_PP_VALIDO[1]).sum():,}")
print(f"no numéricos  : {_v.isna().sum() - df_raw['ValorObservado'].isna().sum():,}")


## 3. Limpieza

Cuatro operaciones, en este orden:

1. **Normalización de texto** — `.str.strip()` para los extremos y colapso de espacios internos
   múltiples (`GALERAZAMBA  - AUT` → `GALERAZAMBA - AUT`). Sin esto, agrupar por nombre produce
   categorías espurias.
2. **Parseo de fecha** — después de limpiar, para que espacios sobrantes no rompan el parseo.
   Se pasa `format=` explícito: sin él pandas infiere fila por fila y es órdenes de magnitud más lento.
3. **Control de rango** — fuera de `RANGO_PP_VALIDO` pasa a `NaN` en vez de eliminarse, preservando
   la marca temporal del registro fallido. Se añaden dos banderas: `qc_fuera_rango` (anulado) y
   `qc_sospechoso` (conservado, pero por encima de `UMBRAL_PP_SOSPECHOSO`).
4. **Desduplicación** — por `(CodigoEstacion, FechaObservacion)`, **sin** incluir `CodigoSensor`.

El punto 4 pesa más aquí que en temperatura. Con un promedio, un registro duplicado apenas mueve el
resultado; con una suma, **lo duplica**. Si el mismo instante aparece bajo dos códigos de sensor
(`Precipitacion` y `GPRS - Precipitacion` son vías de transmisión, no instrumentos distintos), incluir
`CodigoSensor` en la clave deja pasar el duplicado y el acumulado mensual sale inflado.

La distinción entre `NaN` y `0.0` es crítica en esta variable: `0.0` significa "no llovió",
`NaN` significa "no se sabe". Confundirlos convierte huecos de instrumentación en sequías.


In [ ]:
def limpiar(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Normaliza texto, parsea fechas, aplica QC de rango y desduplica."""
    df = df_raw.copy()

    # 1. texto: extremos + espacios internos colapsados
    for c in COLS_TEXTO:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

    # 2. fecha
    df["FechaObservacion"] = pd.to_datetime(df["FechaObservacion"], format=FORMATO_FECHA)

    # 3. rango físico → NaN (no se borra la fila) + banderas de QC
    lo, hi = RANGO_PP_VALIDO
    df["ValorObservado"] = pd.to_numeric(df["ValorObservado"], errors="coerce")
    fuera = ~df["ValorObservado"].between(lo, hi)
    df["qc_fuera_rango"] = fuera.fillna(True)
    df.loc[fuera, "ValorObservado"] = np.nan
    df["qc_sospechoso"] = df["ValorObservado"] > UMBRAL_PP_SOSPECHOSO

    # 4. desduplicación determinista: se ordena antes para que keep="first" sea reproducible
    df = (
        df.sort_values(["CodigoEstacion", "FechaObservacion", "CodigoSensor"], kind="stable")
        .drop_duplicates(subset=["CodigoEstacion", "FechaObservacion"], keep="first")
        .reset_index(drop=True)
    )
    return df


df = limpiar(df_raw)

print(f"crudo        : {len(df_raw):,}")
print(f"limpio       : {len(df):,}  ({len(df_raw) - len(df):,} duplicados eliminados)")
print(f"anulados QC  : {int(df['qc_fuera_rango'].sum()):,} fuera de {RANGO_PP_VALIDO} mm")
print(f"sospechosos  : {int(df['qc_sospechoso'].sum()):,} por encima de {UMBRAL_PP_SOSPECHOSO} mm "
      f"(conservados)")

# Los sospechosos merecen una mirada antes de aceptarlos como aguaceros reales
if df["qc_sospechoso"].any():
    display(
        df.loc[df["qc_sospechoso"],
               ["CodigoEstacion", "NombreEstacion", "FechaObservacion", "ValorObservado"]]
        .sort_values("ValorObservado", ascending=False)
        .head(15)
    )


## 4. Catálogo de estaciones


Un mismo `CodigoEstacion` puede aparecer con **varias grafías de nombre** en el archivo. Agrupar por
`(código, nombre)` fragmenta la misma estación en varias filas del catálogo y, más adelante, en
varias series distintas.

Aquí se elige un **nombre canónico** por código: el más frecuente, con desempate alfabético para que
el resultado sea determinista. Se agrega también el total acumulado por estación, que es el primer
tamiz de plausibilidad: en el Caribe colombiano una estación con años de registro debería estar en
el orden de 800–1 200 mm/año.


In [ ]:
def construir_catalogo(df: pd.DataFrame) -> pd.DataFrame:
    """Una fila por CodigoEstacion, con nombre canónico y coordenadas robustas."""
    # nombre más frecuente por código; desempate alfabético para reproducibilidad
    conteo = (
        df.groupby(["CodigoEstacion", "NombreEstacion"], as_index=False)
        .size()
        .rename(columns={"size": "_n"})
    )
    nombre_canonico = (
        conteo.sort_values(
            ["CodigoEstacion", "_n", "NombreEstacion"],
            ascending=[True, False, True],
            kind="stable",
        )
        .drop_duplicates("CodigoEstacion", keep="first")
        .drop(columns="_n")
    )

    agregados = (
        df.groupby("CodigoEstacion")
        .agg(
            Municipio=("Municipio", lambda s: s.mode().iat[0]),
            ZonaHidrografica=("ZonaHidrografica", lambda s: s.mode().iat[0]),
            # mediana: robusta ante coordenadas truncadas en algunos registros
            Latitud=("Latitud", "median"),
            Longitud=("Longitud", "median"),
            n_variantes_nombre=("NombreEstacion", "nunique"),
            n_sensores=("CodigoSensor", "nunique"),
            n_obs=("ValorObservado", "size"),
            n_validos=("ValorObservado", "count"),
            pp_total_mm=("ValorObservado", "sum"),
            pp_max_intervalo=("ValorObservado", "max"),
            inicio=("FechaObservacion", "min"),
            fin=("FechaObservacion", "max"),
        )
        .reset_index()
    )
    agregados["pp_total_mm"] = agregados["pp_total_mm"].round(1)

    return nombre_canonico.merge(agregados, on="CodigoEstacion", how="left")


catalogo = construir_catalogo(df)
print(f"{len(catalogo)} estaciones únicas")
catalogo.sort_values("n_obs", ascending=False)


## 5. Geometría: ROI y cajas anidadas


In [ ]:
roi = ee.Geometry.Polygon([ROI_COORDS + [ROI_COORDS[0]]])  # cierre explícito del polígono

# Centro y semiejes del bounding box del ROI (cálculo local, sin llamadas a GEE)
_lons = [c[0] for c in ROI_COORDS]
_lats = [c[1] for c in ROI_COORDS]
CX, CY = (min(_lons) + max(_lons)) / 2, (min(_lats) + max(_lats)) / 2
HW, HH = (max(_lons) - min(_lons)) / 2, (max(_lats) - min(_lats)) / 2

BBOXES = {f: (CX - HW * f, CY - HH * f, CX + HW * f, CY + HH * f) for f in FACTORES}
CAJAS = {
    f: ee.Geometry.Rectangle(list(BBOXES[f]), proj="EPSG:4326", geodesic=False)
    for f in FACTORES
}

print(f"Centro del ROI: {CY:.6f}, {CX:.6f}\n")
for f in FACTORES:
    x0, y0, x1, y1 = BBOXES[f]
    ancho = (x1 - x0) * 111.32 * math.cos(math.radians(CY))
    alto = (y1 - y0) * 111.32
    print(f"  {f}x → {ancho:5.2f} km × {alto:5.2f} km   (área ≈ {ancho*alto:5.2f} km²)")


## 6. Clasificación espacial y distancias


La distancia se calcula con **haversine** (esférica) en vez de una aproximación plana. A escalas de
10–150 km la diferencia es de decenas de metros, pero es el número correcto para citar.

Una advertencia que aplica a esta variable más que a la temperatura: la distancia euclídea es un mal
predictor de la representatividad pluviométrica. Los campos de temperatura son suaves a escala
regional; los de precipitación convectiva tienen longitudes de correlación de pocos kilómetros. Una
estación a 10 km puede registrar 0 mm mientras sobre la laguna cae un aguacero. La distancia sirve
para ordenar candidatas, no para garantizar que la serie sea representativa.


In [ ]:
def dist_haversine_km(lat, lon, lat0: float, lon0: float, R: float = 6371.0088) -> np.ndarray:
    """Distancia esférica desde cada punto hasta (lat0, lon0), en km. Vectorizada."""
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    dphi = np.radians(lat0 - lat)
    dlam = np.radians(lon0 - lon)
    a = (
        np.sin(dphi / 2) ** 2
        + np.cos(np.radians(lat)) * math.cos(math.radians(lat0)) * np.sin(dlam / 2) ** 2
    )
    return 2 * R * np.arcsin(np.sqrt(a))


def clasificar_por_caja(df: pd.DataFrame) -> pd.Series:
    """Nivel de la caja más pequeña que contiene el punto; <NA> si ninguna."""
    nivel = pd.Series(pd.NA, index=df.index, dtype="Int32")
    for f in sorted(FACTORES, reverse=True):  # de mayor a menor: gana la más pequeña
        x0, y0, x1, y1 = BBOXES[f]
        dentro = df["Longitud"].between(x0, x1) & df["Latitud"].between(y0, y1)
        nivel = nivel.mask(dentro, f)
    return nivel


estaciones = catalogo.copy()
estaciones["nivel_caja"] = clasificar_por_caja(estaciones)
estaciones["dist_km"] = dist_haversine_km(
    estaciones["Latitud"], estaciones["Longitud"], CY, CX
).round(2)
estaciones = estaciones.sort_values("dist_km", kind="stable").reset_index(drop=True)

n_dentro = int(estaciones["nivel_caja"].notna().sum())
print(f"Estaciones dentro de alguna caja: {n_dentro}")
print(f"Estación más cercana: {estaciones['dist_km'].iat[0]:.2f} km\n")

estaciones[["CodigoEstacion", "NombreEstacion", "Municipio", "nivel_caja",
            "dist_km", "n_obs", "pp_total_mm"]].head(10)


### Sitios físicos únicos

La red del IDEAM registra a veces el mismo instrumento bajo dos códigos (por ejemplo, uno con sufijo
de transmisión telemétrica tipo `TX GPRS`). Contarlos como dos estaciones sería un error que un
revisor familiarizado con la red notaría; y en precipitación, además, produciría dos series
idénticas presentadas como validación cruzada independiente.

Se colapsan las entradas de catálogo que caen en las mismas coordenadas redondeadas a
`TOL_SITIO` decimales (≈ 110 m).


In [ ]:
def colapsar_sitios(estaciones: pd.DataFrame, tol: int = TOL_SITIO) -> pd.DataFrame:
    """Colapsa entradas de catálogo que corresponden al mismo sitio físico."""
    e = estaciones.sort_values("dist_km", kind="stable").copy()
    e["_la"] = e["Latitud"].round(tol)
    e["_lo"] = e["Longitud"].round(tol)

    g = e.groupby(["_la", "_lo"])["CodigoEstacion"]
    e["n_entradas_catalogo"] = g.transform("size")
    e["codigos_del_sitio"] = g.transform(lambda s: [list(s)] * len(s))

    return (
        e.drop_duplicates(["_la", "_lo"], keep="first")
        .drop(columns=["_la", "_lo"])
        .reset_index(drop=True)
    )


sitios = colapsar_sitios(estaciones)
print(f"{len(estaciones)} entradas de catálogo → {len(sitios)} sitios físicos\n")

sitios[["CodigoEstacion", "NombreEstacion", "dist_km",
        "n_entradas_catalogo", "codigos_del_sitio"]].head(8)


## 7. Mapa


El mapa se construye desde cero dentro de una función. Si las capas se añaden a un objeto
preexistente, se acumulan entre ejecuciones y aparecen marcadores duplicados superpuestos.


In [ ]:
import ipyleaflet
from ipywidgets import HTML


def construir_mapa(sitios_sel: pd.DataFrame, zoom: int = 11) -> geemap.Map:
    """Mapa con el ROI, las cajas anidadas y las estaciones seleccionadas."""
    m = geemap.Map(center=[CY, CX], zoom=zoom)
    m.add_basemap("SATELLITE")

    # ROI
    m.addLayer(
        ee.Image().paint(ee.FeatureCollection([ee.Feature(roi)]), 0, 3),
        {"palette": "00FFFF"}, "Laguna (ROI)",
    )

    # Cajas, de la mayor a la menor para que la roja quede encima
    for f in sorted(FACTORES, reverse=True):
        m.addLayer(
            ee.Image().paint(ee.FeatureCollection([ee.Feature(CAJAS[f])]), 0, 3),
            {"palette": COLORES_CAJA[f]}, f"Caja {f}x",
        )

    # Estaciones: línea al centro del ROI + etiqueta de distancia + marcador
    paleta = ["#FF00FF", "#00FFAA", "#FFAA00", "#AA66FF"]
    for i, (_, r) in enumerate(sitios_sel.iterrows()):
        lat_e, lon_e, color = r["Latitud"], r["Longitud"], paleta[i % len(paleta)]

        m.add(ipyleaflet.Polyline(
            locations=[(CY, CX), (lat_e, lon_e)],
            color=color, weight=3, opacity=0.9, fill=False,
        ))

        # Etiqueta a distinta fracción del trayecto para que no se encimen
        frac = 0.5 - 0.12 * i
        m.add(ipyleaflet.Marker(
            location=(CY + (lat_e - CY) * frac, CX + (lon_e - CX) * frac),
            draggable=False,
            icon=ipyleaflet.DivIcon(
                html=(
                    f'<div style="background:{color};color:#000;padding:3px 8px;'
                    f'border-radius:4px;font-weight:bold;font-size:13px;'
                    f'white-space:nowrap;border:1px solid #000;">'
                    f'{r["d_borde_km"]:.2f} km</div>'
                ),
                icon_size=[0, 0],
            ),
        ))

        m.add_marker(
            location=(lat_e, lon_e),
            popup=HTML(
                f"<b>{r['NombreEstacion']}</b><br>"
                f"Código: {r['CodigoEstacion']}<br>"
                f"{r['Municipio']}<br>"
                f"Al centro del ROI: {r['d_centro_km']:.2f} km<br>"
                f"Al borde del ROI: {r['d_borde_km']:.2f} km<br>"
                f"Observaciones: {r['n_obs']:,}<br>"
                f"Acumulado del registro: {r['pp_total_mm']:,.1f} mm"
            ),
        )

    m.add_legend(title="Cajas", legend_dict={f"{f}x": "#" + COLORES_CAJA[f] for f in FACTORES})

    # Encuadre que cubre ROI y estaciones
    lats = [CY] + sitios_sel["Latitud"].tolist()
    lons = [CX] + sitios_sel["Longitud"].tolist()
    m.center = ((min(lats) + max(lats)) / 2, (min(lons) + max(lons)) / 2)
    return m


## 8. Selección de estaciones y serie temporal


Se reportan dos distancias: al **centro** del ROI y a su **borde**. La segunda es la que conviene
citar cuando se pregunte "¿a qué distancia del área de estudio?", porque no depende del tamaño del
polígono.


In [ ]:
# Distancias geodésicas exactas vía GEE, solo para las candidatas seleccionadas
seleccion = sitios.head(N_ESTACIONES).copy()
p_centro = ee.Geometry.Point([CX, CY])

d_centro, d_borde = [], []
for _, r in seleccion.iterrows():
    p = ee.Geometry.Point([r["Longitud"], r["Latitud"]])
    d_centro.append(p_centro.distance(p, maxError=1).getInfo() / 1000)
    d_borde.append(roi.distance(p, maxError=1).getInfo() / 1000)

seleccion["d_centro_km"] = np.round(d_centro, 2)
seleccion["d_borde_km"] = np.round(d_borde, 2)

for _, r in seleccion.iterrows():
    print(f"{r['NombreEstacion'][:38]:<40} centro: {r['d_centro_km']:6.2f} km   "
          f"borde: {r['d_borde_km']:6.2f} km")


In [ ]:
Map = construir_mapa(seleccion)
Map


In [ ]:
# Serie cruda de las estaciones seleccionadas.
# Se incluyen TODOS los códigos de cada sitio físico y se les asigna el nombre canónico DEL SITIO,
# para que un mismo pluviómetro no se parta en dos series.
mapa_sitio = (
    seleccion[["codigos_del_sitio", "NombreEstacion"]]
    .explode("codigos_del_sitio")
    .rename(columns={"codigos_del_sitio": "CodigoEstacion", "NombreEstacion": "Estacion"})
)
print("Códigos incluidos:", mapa_sitio["CodigoEstacion"].tolist())

serie = (
    df.merge(mapa_sitio, on="CodigoEstacion", how="inner")
    [["Estacion", "CodigoEstacion", "FechaObservacion", "ValorObservado",
      "qc_fuera_rango", "qc_sospechoso"]]
    .sort_values(["Estacion", "FechaObservacion"], kind="stable")
    .reset_index(drop=True)
)

# Un mismo sitio con dos códigos puede repetir el instante: se desduplica otra vez a nivel de SITIO.
n_antes = len(serie)
serie = serie.drop_duplicates(subset=["Estacion", "FechaObservacion"], keep="first").reset_index(drop=True)
if n_antes != len(serie):
    print(f"⚠  {n_antes - len(serie):,} instantes repetidos entre códigos del mismo sitio, eliminados")

print(f"{len(serie):,} registros en {serie['Estacion'].nunique()} sitios")
serie.head()


## 9. ¿Acumulado por intervalo o acumulado corrido?

Antes de sumar nada hay que responder una pregunta que el archivo no contesta explícitamente:
`ValorObservado` puede ser la lámina caída **en ese intervalo** (lo habitual en la red automática
del IDEAM con sensor `0240`) o un **contador acumulado** que se reinicia periódicamente. Si es lo
segundo y se suma, el total sale inflado en varios órdenes de magnitud.

La prueba es barata: en un contador acumulado la secuencia dentro del día es **monótona no
decreciente**. Se evalúan solo los días con al menos tres registros positivos (en días secos la
serie es constante en cero y sería monótona de forma trivial, sin informar nada).

**Lectura del resultado:** un porcentaje cercano a 100 % indica contador acumulado y obliga a
diferenciar antes de agregar. Un porcentaje bajo (típicamente < 30 %, porque los aguaceros suben y
bajan) confirma que cada valor es independiente y se puede sumar directamente.


In [ ]:
def diagnostico_acumulacion(serie: pd.DataFrame) -> pd.DataFrame:
    """% de días cuya secuencia intradiaria es monótona no decreciente (señal de contador)."""
    s = serie.dropna(subset=["ValorObservado"]).sort_values(
        ["Estacion", "FechaObservacion"], kind="stable").copy()
    s["dia"] = s["FechaObservacion"].dt.floor("D")
    s["_d"] = s.groupby(["Estacion", "dia"])["ValorObservado"].diff()

    r = (
        s.groupby(["Estacion", "dia"])
        .agg(n_pos=("ValorObservado", lambda x: int((x > 0).sum())), min_d=("_d", "min"))
        .reset_index()
    )
    r = r[r["n_pos"] >= 3]
    if r.empty:
        print("No hay días con suficientes registros positivos para evaluar.")
        return pd.DataFrame(columns=["Estacion", "dias_evaluados", "pct_monotonos"])

    r["monotono"] = r["min_d"].fillna(0) >= 0
    return (
        r.groupby("Estacion")
        .agg(dias_evaluados=("monotono", "size"),
             pct_monotonos=("monotono", lambda x: round(100 * float(x.mean()), 1)))
        .reset_index()
    )


acum = diagnostico_acumulacion(serie)
display(acum)

if not acum.empty and (acum["pct_monotonos"] > 90).any():
    print("⚠  Alguna estación parece registrar un ACUMULADO CORRIDO. Diferenciar antes de sumar.")
else:
    print("✓ Los valores se comportan como lámina por intervalo: la suma directa es correcta.")


## 10. Paso de muestreo real y huecos del registro

**La serie se conserva en la resolución nativa de las estaciones.** No se remuestrea a hora, día ni
mes: `serie` mantiene un registro por observación tal como lo entrega el IDEAM. Lo que sigue no
transforma los datos, solo los caracteriza.

**Paso de muestreo, por estación y por año.** Una misma estación cambia de frecuencia a lo largo de
su historia (10 min en unos períodos, horaria en otros). Estimar un único paso para toda la serie da
un número equivocado justo en los años de transición. El `diff()` se calcula **dentro de cada
estación**: hacerlo sobre la tabla completa mezclaría el último registro de una con el primero de la
siguiente.

Conocer el paso importa por dos razones, aunque no se agregue nada:

- Es lo que permite convertir una lámina por intervalo (mm) en una **intensidad** (mm/h). Sin él,
  0.5 mm de un registro de 10 minutos y 0.5 mm de uno horario parecen el mismo dato y son
  intensidades que difieren en un factor 6.
- Es el denominador de la completitud y el criterio para distinguir un hueco real de un intervalo
  normal.


In [ ]:
def paso_por_anio(serie: pd.DataFrame) -> pd.DataFrame:
    """Paso de muestreo mediano (segundos) por estación y año."""
    s = serie.sort_values(["Estacion", "FechaObservacion"], kind="stable").copy()
    s["_d"] = s.groupby("Estacion")["FechaObservacion"].diff().dt.total_seconds()
    s["anio"] = s["FechaObservacion"].dt.year

    p = (
        s.groupby(["Estacion", "anio"])["_d"].median()
        .reset_index().rename(columns={"_d": "paso_s"})
    )
    p["paso_s"] = p["paso_s"].where(p["paso_s"] > 0).fillna(PASO_FALLBACK_S)
    p["paso_min"] = (p["paso_s"] / 60).round(1)
    return p


def anexar_paso(serie: pd.DataFrame, pasos: pd.DataFrame) -> pd.DataFrame:
    """Añade a cada registro el paso vigente y su intensidad equivalente en mm/h."""
    s = serie.copy()
    s["anio"] = s["FechaObservacion"].dt.year
    s = s.merge(pasos[["Estacion", "anio", "paso_s"]], on=["Estacion", "anio"], how="left")
    s["paso_s"] = s["paso_s"].fillna(PASO_FALLBACK_S)
    s["intensidad_mmh"] = (s["ValorObservado"] * 3600.0 / s["paso_s"]).round(2)
    return s.drop(columns="anio")


pasos = paso_por_anio(serie)
serie_pp = anexar_paso(serie, pasos)   # serie nativa enriquecida, misma cantidad de filas

print(f"serie: {len(serie):,} registros  →  serie_pp: {len(serie_pp):,} registros "
      f"(sin remuestreo)\n")
print("Paso de muestreo mediano (minutos) por año")
display(pasos.pivot(index="anio", columns="Estacion", values="paso_min"))


In [ ]:
def completitud_anual(serie: pd.DataFrame, pasos: pd.DataFrame) -> pd.DataFrame:
    """Cuántos registros hay frente a los esperados al paso vigente, año por año."""
    s = serie.assign(anio=serie["FechaObservacion"].dt.year)
    out = (
        s.groupby(["Estacion", "anio"])
        .agg(n_registros=("ValorObservado", "size"),
             n_validos=("ValorObservado", "count"),
             inicio=("FechaObservacion", "min"),
             fin=("FechaObservacion", "max"))
        .reset_index()
        .merge(pasos, on=["Estacion", "anio"], how="left")
    )
    span = (out["fin"] - out["inicio"]).dt.total_seconds()
    dias = out["anio"].map(lambda a: pd.Timestamp(int(a), 12, 31).dayofyear)

    # Dos medidas distintas que no hay que confundir:
    #   pct_anio_cubierto : qué parte del año calendario abarca el registro
    #   pct_completitud   : dentro de ese tramo, qué fracción de registros esperados existe
    out["n_esperados"] = (span / out["paso_s"] + 1).round()
    out["pct_completitud"] = (out["n_registros"] / out["n_esperados"] * 100).round(1)
    out["pct_anio_cubierto"] = (span / (dias * 86400) * 100).round(1)
    return out


def detectar_huecos(serie: pd.DataFrame, pasos: pd.DataFrame,
                    factor: int = FACTOR_HUECO) -> pd.DataFrame:
    """Interrupciones del registro: saltos mayores que `factor` veces el paso vigente."""
    s = anexar_paso(serie.sort_values(["Estacion", "FechaObservacion"], kind="stable"), pasos)
    s["_prev"] = s.groupby("Estacion")["FechaObservacion"].shift()
    s["_gap"] = (s["FechaObservacion"] - s["_prev"]).dt.total_seconds()

    h = s[s["_gap"] > factor * s["paso_s"]].copy()
    h["duracion_h"] = (h["_gap"] / 3600).round(2)
    h["duracion_d"] = (h["_gap"] / 86400).round(2)
    return (
        h.rename(columns={"_prev": "ultimo_dato", "FechaObservacion": "siguiente_dato"})
        [["Estacion", "ultimo_dato", "siguiente_dato", "duracion_h", "duracion_d"]]
        .sort_values("duracion_h", ascending=False)
        .reset_index(drop=True)
    )


completitud = completitud_anual(serie, pasos)
display(completitud[["Estacion", "anio", "paso_min", "n_registros", "n_esperados",
                     "pct_completitud", "pct_anio_cubierto", "n_validos"]])

huecos = detectar_huecos(serie, pasos)
print(f"\n{len(huecos):,} interrupciones (> {FACTOR_HUECO}× el paso). "
      f"Las 10 más largas:")
display(huecos.head(10))


## 11. Descriptores sobre la serie nativa

Los descriptores de temperatura (media, σ, rango) no sirven aquí: en una serie de intervalos de
10 minutos la inmensa mayoría de los valores es 0, la media es del orden de 0.01 mm y la σ está
dominada por unos pocos aguaceros. Sobre la resolución nativa los descriptores informativos son:

- **Total acumulado** del registro y su equivalente medio anual (con la advertencia de que un total
  sobre un registro incompleto no es un total real; leerlo junto a la tabla de completitud).
- **Fracción de intervalos húmedos** (≥ `MM_INTERVALO_HUMEDO`), que es la frecuencia de lluvia a la
  resolución del instrumento.
- **Intensidad condicional**: media y percentiles calculados *solo sobre intervalos húmedos*, tanto
  en mm por intervalo como en mm/h. La segunda es la única comparable entre estaciones con pasos
  distintos.
- **Máximos**: la lámina mayor en un intervalo y la intensidad instantánea máxima.


In [ ]:
def descriptores_nativos(serie_pp: pd.DataFrame) -> pd.DataFrame:
    """Descriptores de precipitación calculados sobre los registros originales."""
    humedos = serie_pp[serie_pp["ValorObservado"] >= MM_INTERVALO_HUMEDO]

    base = (
        serie_pp.groupby("Estacion")
        .agg(inicio=("FechaObservacion", "min"), fin=("FechaObservacion", "max"),
             n_registros=("ValorObservado", "size"), n_validos=("ValorObservado", "count"),
             pp_total_mm=("ValorObservado", "sum"),
             pp_max_intervalo=("ValorObservado", "max"),
             intensidad_max_mmh=("intensidad_mmh", "max"))
    )
    cond = (
        humedos.groupby("Estacion")
        .agg(n_humedos=("ValorObservado", "size"),
             media_humedo_mm=("ValorObservado", "mean"),
             p95_humedo_mm=("ValorObservado", lambda x: x.quantile(0.95)),
             intensidad_media_mmh=("intensidad_mmh", "mean"),
             intensidad_p95_mmh=("intensidad_mmh", lambda x: x.quantile(0.95)))
    )

    out = base.join(cond).reset_index()
    out["pct_intervalos_humedos"] = (out["n_humedos"] / out["n_validos"] * 100).round(2)
    out["anios_registro"] = (
        (out["fin"] - out["inicio"]).dt.total_seconds() / (365.25 * 86400)).round(2)
    out["pp_media_anual_mm"] = (out["pp_total_mm"] / out["anios_registro"]).round(1)

    for c in ["pp_total_mm", "pp_max_intervalo", "intensidad_max_mmh", "media_humedo_mm",
              "p95_humedo_mm", "intensidad_media_mmh", "intensidad_p95_mmh"]:
        out[c] = out[c].round(2)
    return out


descriptores = descriptores_nativos(serie_pp)
display(descriptores[[
    "Estacion", "inicio", "fin", "anios_registro", "n_registros", "n_validos",
    "pp_total_mm", "pp_media_anual_mm", "n_humedos", "pct_intervalos_humedos",
    "media_humedo_mm", "p95_humedo_mm", "pp_max_intervalo",
    "intensidad_media_mmh", "intensidad_p95_mmh", "intensidad_max_mmh",
]])

print("\nAviso: `pp_media_anual_mm` divide el acumulado entre los años del registro y NO corrige")
print("por los huecos de la tabla de completitud. Es un orden de magnitud, no un valor citable.")


## 12. Visualización


Las tres figuras se construyen sobre los registros originales. En la primera, cada línea vertical es
**un registro**: no hay barras diarias ni promedios móviles que suavicen la señal. Con esta variable
eso es una ventaja, porque la estructura interesante (aguaceros cortos e intensos) es justamente la
que un remuestreo esconde.

Los huecos se dibujan como bandas grises. Sin ellos, una interrupción de instrumentación se lee
igual que una racha seca.


In [ ]:
PALETA = ["#2E6FBD", "#2E8B57", "#C0562E", "#6A4C93"]
nombres = serie_pp["Estacion"].drop_duplicates().tolist()

fig, axes = plt.subplots(len(nombres), 1, figsize=(15, 3.6 * len(nombres)), sharex=True)
axes = np.atleast_1d(axes)

for ax, nombre, color in zip(axes, nombres, PALETA):
    sub = serie_pp[serie_pp["Estacion"] == nombre]
    hum = sub[sub["ValorObservado"] >= MM_INTERVALO_HUMEDO]
    d = descriptores.set_index("Estacion").loc[nombre]

    # Un trazo vertical por registro con lluvia (resolución nativa, sin agregar)
    ax.vlines(hum["FechaObservacion"], 0, hum["ValorObservado"],
              color=color, lw=0.4, alpha=0.8)

    # Huecos del registro como bandas
    for _, hh in huecos[huecos["Estacion"] == nombre].iterrows():
        ax.axvspan(hh["ultimo_dato"], hh["siguiente_dato"],
                   color="#999", alpha=0.22, lw=0)

    ax.axhline(d["p95_humedo_mm"], color="#666", lw=0.8, ls=":", alpha=0.9)
    ax.annotate(f'p95 {d["p95_humedo_mm"]:.1f}', xy=(1.0, d["p95_humedo_mm"]),
                xycoords=("axes fraction", "data"), xytext=(4, 0),
                textcoords="offset points", va="center", ha="left", fontsize=8, color="#333")

    caja = (f'registros    = {int(d["n_registros"]):,}\n'
            f'húmedos      = {int(d["n_humedos"]):,} ({d["pct_intervalos_humedos"]:.2f} %)\n'
            f'total        = {d["pp_total_mm"]:,.1f} mm\n'
            f'media húmedo = {d["media_humedo_mm"]:.2f} mm/int\n'
            f'máx intervalo= {d["pp_max_intervalo"]:.1f} mm\n'
            f'máx intens.  = {d["intensidad_max_mmh"]:.1f} mm/h')
    ax.text(0.012, 0.96, caja, transform=ax.transAxes, fontsize=8.5,
            va="top", ha="left", family="monospace",
            bbox=dict(boxstyle="round,pad=0.45", facecolor="white",
                      edgecolor=color, alpha=0.9, linewidth=1.2))

    ax.set_title(nombre, fontsize=10, loc="left", fontweight="bold")
    ax.set_ylabel("mm / intervalo")
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)

axes[-1].set_xlabel("Fecha")
fig.suptitle("Precipitación en resolución nativa — un trazo por registro\n"
             "(bandas grises: interrupciones del registro)", fontsize=12, y=0.995)
plt.tight_layout(rect=[0, 0, 0.94, 0.97])
plt.show()


In [ ]:
serie_pp

## 13. Limitaciones a declarar en el manuscrito

**Ausencia de estaciones en el área de estudio.** Verificar en la sección 6 cuántas estaciones caen
dentro de las cajas anidadas (en el análisis de temperatura, ninguna lo hacía ni en la mayor,
≈ 3.4 × 3.9 km). La distancia al borde del ROI de la estación más próxima es el número a reportar.

**Representatividad espacial, más severa que en temperatura.** La precipitación en el Caribe
colombiano es predominantemente convectiva: celdas de pocos kilómetros, de vida corta, con
gradientes muy fuertes. Una serie pluviométrica a ~10 km del ROI describe el *régimen* estacional e
interanual, pero no permite afirmar que llovió sobre la laguna en una fecha concreta. Cualquier
análisis evento a evento excede lo que estos datos soportan.

**Sesgo espacial de la red.** Revisar la distribución azimutal de las estaciones seleccionadas
respecto al ROI: si todas quedan hacia un mismo cuadrante, cualquier gradiente en la dirección
opuesta queda sin muestrear.

**Los totales no están corregidos por huecos.** `pp_total_mm` y `pp_media_anual_mm` suman lo que hay;
la tabla de completitud y la de huecos dicen cuánto falta. Un año con 71 % de completitud subestima
el total en una magnitud desconocida (no proporcional, porque los huecos no se reparten
uniformemente entre estación seca y húmeda). Citar ambos números juntos o ninguno.

**Heterogeneidad de muestreo entre estaciones.** Al conservar la resolución nativa, las estaciones no
son directamente comparables registro a registro: 0.5 mm en 10 minutos y 0.5 mm en una hora son
intensidades distintas. Para comparar, usar `intensidad_mmh` (o el anexo de la sección 15), nunca la
columna `ValorObservado` cruda entre estaciones de distinto paso.

**Umbrales empíricos.** `RANGO_PP_VALIDO`, `UMBRAL_PP_SOSPECHOSO`, `MM_INTERVALO_HUMEDO` y
`FACTOR_HUECO` son elecciones de este trabajo, no estándares del IDEAM. Deben declararse
explícitamente y, en lo posible, acompañarse de un análisis de sensibilidad.

**Naturaleza del registro.** La sección 9 verifica empíricamente que los valores son láminas por
intervalo y no un contador acumulado. Esa verificación debe rehacerse si se cambia de fuente, de
sensor o de período de descarga.